# Spoken dialogue: listen, think, speak

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/espnet/notebook/blob/master/Demos/sds_demo.ipynb) [![sds_demo](https://github.com/espnet/notebook/actions/workflows/sds_demo.yml/badge.svg)](https://github.com/espnet/notebook/actions/workflows/sds_demo.yml)

Speech in, speech out, with a language model in between. Each stage is one
class of `espnet2.sds`, the module ESPnet's spoken dialogue system is built
from, so any of them can be swapped by changing a model tag. ESPnet does the
listening and the speaking; the thinking is a small instruct model from the
Hugging Face Hub behind the same kind of wrapper.

CPU throughout. A couple of minutes, most of it the first download. How long
each stage takes, with voice activity detection in front and a microphone at
the end, is [`sds_latency_demo.ipynb`](sds_latency_demo.ipynb).

## Install

In [ ]:
%pip install -q "espnet[tts]==202611.post1" espnet_model_zoo librosa transformers

## Somebody says something

A real recording rather than a synthesised one: a dialogue system meets
real voices, and reading back its own speech would flatter it.

In [ ]:
import librosa
from IPython.display import Audio, display

!wget -q -O turn.wav https://github.com/espnet/espnet/raw/master/test_utils/ctc_align_test.wav
speech, rate = librosa.load("turn.wav", sr=16000)
display(Audio(speech, rate=rate))

## Listen

`OWSMCTCModel`: OWSM-CTC, the same model as [`asr_demo.ipynb`](asr_demo.ipynb),
read off its CTC head in one pass over a 30 s window.

In [ ]:
from espnet2.sds.asr.owsm_ctc_asr import OWSMCTCModel

asr = OWSMCTCModel(tag="espnet/owsm_ctc_v4_1B", device="cpu")
heard = asr.forward(speech)
print("heard:", heard)

## Think

`HuggingFaceLLM` takes any chat model on the Hub. A 0.5B instruct model is
small enough to answer on a CPU.

Chosen by trying four of them on this exact turn. The smaller SmolLM2-360M,
the five-times-larger SmolLM2-1.7B and Qwen3-0.6B all hand the sentence back
reworded; this one answers it. Size was not the axis.

In [ ]:
from espnet2.sds.llm.hugging_face_llm import HuggingFaceLLM

llm = HuggingFaceLLM(tag="Qwen/Qwen2.5-0.5B-Instruct", device="cpu", dtype="float32",
                     max_new_tokens=40)
conversation = [
    {"role": "system", "content": "You are a helpful assistant in a spoken conversation. Reply to what the user just said in one short, natural sentence."},
    {"role": "user", "content": heard},
]
reply = llm.forward(conversation)
print("reply:", reply)

## Speak

`ESPnetTTSModel`: the same VITS as [`tts_demo.ipynb`](tts_demo.ipynb),
reading the answer back.

In [ ]:
from espnet2.sds.tts.espnet_tts import ESPnetTTSModel

tts = ESPnetTTSModel(tag="espnet/kan-bayashi_ljspeech_vits", device="cpu")
fs, wav = tts.forward(reply)
display(Audio(wav, rate=fs))

## Where next

- **How long each stage takes**, with voice activity detection in front and
  a microphone loop at the end: [`sds_latency_demo.ipynb`](sds_latency_demo.ipynb)
- **The app**: [`egs2/TEMPLATE/sds1`](https://github.com/espnet/espnet/tree/master/egs2/TEMPLATE/sds1),
  a Gradio app you run locally, with an end-to-end option beside the cascade
  and latency and quality measured while you talk
- **End-to-end**, no cascade: `espnet2.sds.end_to_end` runs Mini-Omni
- **A bigger brain**: any chat model on the Hub goes in `HuggingFaceLLM`;
  half a billion parameters is what a free CPU answers with